# Forgetting to Refuse — Phase 1 on Colab

Runtime → **Change runtime type → A100** (L4 works for Phase 1 only; training needs the A100).

What this does: mounts Drive, clones the repo, builds the environment on local disk (~10 min, no compiles),
keeps the big downloads (~36 GB) on Drive so they survive the runtime, then runs the Phase-1 checklist and
saves every log to **`MyDrive/ftr/logs/phase1/`**. The last cell pushes the logs + `requirements.txt` to the repo.

**Before running:** add two Colab secrets (🔑 icon in the left bar): `GH_TOKEN` (GitHub token with `repo` scope,
for the private repo) — nothing else.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
os.environ['W'] = '/content/ftr'                       # local, ephemeral: venv + clones
os.environ['DATA'] = '/content/drive/MyDrive/ftr'       # persistent: hf cache, runs, data, logs
os.environ['REPO'] = '/content/ftr/repo'
os.environ['CUDA'] = 'cu118'                            # prebuilt flash-attn wheel, no compile
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.makedirs(os.environ['DATA'] + '/logs/phase1', exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
%%bash
set -e
mkdir -p $W
if [ ! -d $REPO ]; then git clone -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git $REPO; fi
git -C $REPO pull -q
git -C $REPO log --oneline | head -1

## Environment (~10 min first time; downloads ~36 GB to Drive once)

In [ ]:
%%bash
set -e
bash $REPO/scripts/setup_pod.sh 2>&1 | tee $DATA/logs/phase1/00_setup.log | tail -5
source $W/env.sh && pip freeze > $DATA/requirements.txt && echo "requirements.txt written

## Phase 1 checklist
Each cell writes its log to Drive. Green = day 1 of PLAN.md is real. Red = paste the log back to Claude.

In [ ]:
%%bash
source $W/env.sh
pytest tests/test_env.py -m gpu -v 2>&1 | tee $DATA/logs/phase1/01_test_env.log | tail -25

In [ ]:
%%bash
source $W/env.sh
pytest tests/test_fixtures.py -m gpu -v 2>&1 | tee $DATA/logs/phase1/02_test_fixtures.log | tail -20

In [ ]:
%%bash
source $W/env.sh
pytest tests/test_parity.py -m gpu -v 2>&1 | tee $DATA/logs/phase1/03_test_parity.log | tail -20

In [ ]:
%%bash
source $W/env.sh
python -m ftr.rollout --ckpt $FTR_P_DIR --suite obstacle_avoidance_human --tasks 3 --states 0-1 \
  --classes harmful benign blank --template-split test --video 3 --out $DATA/runs/smoke \
  2>&1 | tee $DATA/logs/phase1/04_smoke_rollout.log | tail -15
ls -la $DATA/runs/smoke $DATA/runs/smoke/videos 2>/dev/null

## Save results to the repo
Copies the logs and `requirements.txt` into the repo and pushes, so Claude can read them from GitHub.
Videos and Parquet stay on Drive under `MyDrive/ftr/runs/smoke/`.

In [ ]:
%%bash
set -e
cd $REPO
mkdir -p logs/phase1
cp $DATA/logs/phase1/*.log logs/phase1/
cp $DATA/requirements.txt requirements.txt
git config user.email "rq11@rice.edu" && git config user.name "Ryan Quinlivan"
git add logs/phase1 requirements.txt
git commit -qm "Phase 1 logs from Colab ($(nvidia-smi --query-gpu=name --format=csv,noheader))" || echo "nothing to commit"
git push -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git HEAD:main && echo pushed